# Mutational co-occurrence & mutual exclusivity
### Study + cancer type, sample/patient level, mutation/altered — with significance-colored heatmaps

**MSK CBSP Hackathon 2026 — team veggenes**

Pick **any study**, optionally restrict to **one cancer type** (e.g. *Lung Adenocarcinoma*), count at the
**sample** or **patient** level over **mutations only** or **all alterations** (mutation + CNA + fusion), choose any
genes for the two axes, and get a co-occurrence / mutual-exclusivity heatmap:

* **Left panel** — raw count of cases altered in **both** genes of a pair (diagonal = single-gene count).
* **Right panel** — **log₂ odds ratio** from a per-pair **Fisher's exact test**: red = tend to **co-occur**,
  blue = tend to be **mutually exclusive**. A **★** marks pairs significant at **FDR < 0.05** (Benjamini–Hochberg).

> **Why a statistic, not just counts?** Two genes can co-occur in many cases simply because each is individually
> frequent. The odds ratio + test asks whether they co-occur (or avoid each other) *more than expected by chance* —
> distinguishing a real relationship (e.g. KRAS/EGFR exclusivity in LUAD) from a frequency artifact.

Defaults (`LEVEL="sample"`, `ALTERATIONS="altered"`) **reproduce cBioPortal's Mutual Exclusivity tab**.
Data via the [cBioPortal REST API](https://www.cbioportal.org/api) (`requests`).

In [15]:
# First run only (uncomment). On the 'veggenes' kernel these are already installed.
# !pip install pandas numpy scipy statsmodels plotly requests ipywidgets

from IPython.display import display
import requests
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import plotly.graph_objects as go
from plotly.subplots import make_subplots

CBIO     = "https://www.cbioportal.org/api"
JSON_HDR = {"Accept": "application/json", "Content-Type": "application/json"}

# ---- Configure these (or use the pickers below) --------------------
STUDY_ID     = "msk_impact_50k_2026"        # any cBioPortal study id
CANCER_TYPE  = "Lung Adenocarcinoma"        # CANCER_TYPE_DETAILED value, or None for the whole study
GENES_X      = ["KRAS", "TP53", "STK11", "EGFR", "BRAF", "KEAP1"]
GENES_Y      = ["KRAS", "TP53", "STK11", "EGFR", "BRAF", "KEAP1"]   # may differ from GENES_X

# Two toggles control what is counted:
LEVEL        = "sample"     # "sample" or "patient"
ALTERATIONS  = "altered"    # "altered" (mutation OR CNA OR fusion) or "mutation" (mutations only)
#   * LEVEL="sample"  + ALTERATIONS="altered"   -> reproduces cBioPortal's Mutual Exclusivity tab exactly.
#   * LEVEL="patient" + ALTERATIONS="mutation"  -> cleaner standalone stats (independent cases, drivers only).
# --------------------------------------------------------------------

def _get_json(url, **kw):
    r = requests.get(url, headers={"Accept": "application/json"}, timeout=180, **kw)
    r.raise_for_status()
    return r.json()

def _clinical_records(study_id, attribute_id):
    """DataFrame[sampleId, patientId, value] for a SAMPLE-level clinical attribute, paginated.
    Uses the clinical-data list endpoint, which reliably returns a list of records
    (a huge pageSize, or the clinical-data-counts endpoint, can come back as an
    error/object on some deployments)."""
    out, page, size = [], 0, 50_000
    while True:
        cd = _get_json(f"{CBIO}/studies/{study_id}/clinical-data",
                       params={"attributeId": attribute_id, "clinicalDataType": "SAMPLE",
                               "projection": "SUMMARY", "pageSize": size, "pageNumber": page})
        if isinstance(cd, dict):
            raise RuntimeError(f"cBioPortal returned an error instead of clinical data: {cd}")
        out += [(c["sampleId"], c.get("patientId"), c.get("value")) for c in cd]
        if len(cd) < size:
            break
        page += 1
    return pd.DataFrame(out, columns=["sampleId", "patientId", "value"])

def _profiles(study_id):
    """Map alteration kind -> molecularProfileId for the study (mutation / cna / sv)."""
    out = {}
    for p in _get_json(f"{CBIO}/studies/{study_id}/molecular-profiles"):
        t = p["molecularAlterationType"]
        if t == "MUTATION_EXTENDED":
            out["mutation"] = p["molecularProfileId"]
        elif t == "COPY_NUMBER_ALTERATION" and p.get("datatype") == "DISCRETE":
            out["cna"] = p["molecularProfileId"]
        elif t == "STRUCTURAL_VARIANT":
            out["sv"] = p["molecularProfileId"]
    return out

## 1. Choose study and cancer type

Helpers to discover valid `studyId`s and the `CANCER_TYPE_DETAILED` values available within a study (so your filter is guaranteed to match real data).

In [16]:
def find_studies(keyword=""):
    studies = requests.get(f"{CBIO}/studies", params={"projection": "SUMMARY"},
                           headers={"Accept": "application/json"}, timeout=60).json()
    df = pd.DataFrame(studies)
    if keyword:
        k = keyword.lower()
        df = df[df["name"].str.lower().str.contains(k) | df["studyId"].str.lower().str.contains(k)]
    cols = ["studyId", "name", "allSampleCount"]
    return df[cols].sort_values("allSampleCount", ascending=False).reset_index(drop=True)

def list_cancer_types(study_id, top=25):
    """Available CANCER_TYPE_DETAILED values + sample counts for a study."""
    df = _clinical_records(study_id, "CANCER_TYPE_DETAILED")
    s = df["value"].dropna().value_counts().head(top)
    return s.rename_axis("value").reset_index(name="count")

print("Cancer types in", STUDY_ID, "(top 10):")
display(list_cancer_types(STUDY_ID).head(10))

Cancer types in msk_impact_50k_2026 (top 10):


,value,count
0,Lung Adenocarcinoma,6254
1,Breast Invasive Ductal Carcinoma,4320
2,Colon Adenocarcinoma,3694
3,Prostate Adenocarcinoma,3267
4,Pancreatic Adenocarcinoma,2394
5,Bladder Urothelial Carcinoma,1966
6,"Glioblastoma, IDH-Wildtype",1543
7,High-Grade Serous Ovarian Cancer,1429
8,Uterine Endometrioid Carcinoma,1377
9,Rectal Adenocarcinoma,1276


### Optional: pick study + cancer type from dropdowns

Set `SEARCH`, run the cell, choose a study and a cancer type. Selections update `STUDY_ID` and `CANCER_TYPE`. (Skip this cell and set them by hand if you prefer.)

In [17]:
SEARCH = "msk"   # keyword for the study dropdown (matches study name or id)

try:
    import ipywidgets as widgets
    _sdf = find_studies(SEARCH)
    if _sdf.empty:
        print(f"No studies match SEARCH={SEARCH!r}; keeping STUDY_ID={STUDY_ID!r}, CANCER_TYPE={CANCER_TYPE!r}.")
    else:
        _study_opts = [(f"{r['name']}  ({r.studyId}, n={r.allSampleCount})", r.studyId)
                       for _, r in _sdf.iterrows()]
        _study_ids = [o[1] for o in _study_opts]
        _default_study = STUDY_ID if STUDY_ID in _study_ids else _study_opts[0][1]

        _cts = list_cancer_types(_default_study)
        _ct_opts = [("(whole study)", None)] + [(f"{r['value']}  (n={r['count']})", r["value"])
                                                for _, r in _cts.iterrows()]
        _default_ct = CANCER_TYPE if CANCER_TYPE in [o[1] for o in _ct_opts] else None

        _study_dd = widgets.Dropdown(options=_study_opts, value=_default_study,
                                     description="Study:", layout=widgets.Layout(width="95%"))
        _ct_dd = widgets.Dropdown(options=_ct_opts, value=_default_ct,
                                  description="Cancer type:", layout=widgets.Layout(width="95%"))

        # attach observers AFTER initial values are set, so plain execution never
        # clobbers the configured STUDY_ID / CANCER_TYPE -- only real interaction does.
        def _on_study(ch):
            global STUDY_ID
            if ch["new"]:
                STUDY_ID = ch["new"]
                cts = list_cancer_types(STUDY_ID)
                _ct_dd.options = [("(whole study)", None)] + [(f"{r['value']}  (n={r['count']})", r["value"])
                                                              for _, r in cts.iterrows()]
        def _on_ct(ch):
            global CANCER_TYPE
            CANCER_TYPE = ch["new"]
        _study_dd.observe(_on_study, "value")
        _ct_dd.observe(_on_ct, "value")

        display(_study_dd, _ct_dd)
        print(f"Pickers ready. STUDY_ID={STUDY_ID!r}, CANCER_TYPE={CANCER_TYPE!r}. "
              f"Change a dropdown to update; otherwise the values above are used.")
except ImportError:
    print("ipywidgets not installed -- set STUDY_ID / CANCER_TYPE manually.")

Dropdown(description='Study:', layout=Layout(width='95%'), options=(('MSK-IMPACT 50K Clinical Sequencing Cohor…

Dropdown(description='Cancer type:', index=1, layout=Layout(width='95%'), options=(('(whole study)', None), ('…

Pickers ready. STUDY_ID='msk_impact_50k_2026', CANCER_TYPE='Lung Adenocarcinoma'. Change a dropdown to update; otherwise the values above are used.


## 2. Resolve the cohort, validate genes, build the binary matrix

* **Cohort** — if `CANCER_TYPE` is set, we keep only samples whose `CANCER_TYPE_DETAILED` matches; otherwise the study's sequenced samples. This is the step the original notebook was missing — selecting *lung genes* does **not** restrict samples to lung cancer.
* **Gene validation** — symbols are resolved to Entrez IDs via cBioPortal; anything unknown is dropped with a warning, so you can pass arbitrary gene lists safely.
* **Binary matrix** — samples × genes, reindexed over the **whole cohort** so un-mutated samples are counted as 0 (essential for the Fisher 2×2 table and the denominator).

In [18]:
def get_cohort(study_id, cancer_type=None):
    """DataFrame[sampleId, patientId] for the cohort (optionally one cancer type)."""
    df = _clinical_records(study_id, "CANCER_TYPE_DETAILED")
    if cancer_type:
        df = df[df["value"] == cancer_type]
        if df.empty:
            raise ValueError(f"No samples with CANCER_TYPE_DETAILED == {cancer_type!r} in {study_id}. "
                             f"Check list_cancer_types('{study_id}').")
    return df[["sampleId", "patientId"]].drop_duplicates().reset_index(drop=True)

def resolve_genes(symbols):
    res = requests.post(f"{CBIO}/genes/fetch", params={"geneIdType": "HUGO_GENE_SYMBOL"},
                        headers=JSON_HDR, json=list(symbols), timeout=60).json()
    found = {g["hugoGeneSymbol"]: g["entrezGeneId"] for g in res}
    missing = [s for s in symbols if s not in found]
    if missing:
        print(f"WARNING: unknown gene symbol(s) dropped: {missing}")
    return found

def _altered_sample_sets(study_id, entrez_map, sample_ids, alterations):
    """gene -> set(sampleId) altered. 'mutation' = mutations only;
    'altered' = mutation OR CNA (amp/deep-del, ±2) OR structural variant/fusion."""
    e2g = {v: k for k, v in entrez_map.items()}
    sset = set(sample_ids)
    sets = {g: set() for g in entrez_map}
    prof = _profiles(study_id)
    entrez_ids = list(entrez_map.values())

    # mutations
    muts = requests.post(f"{CBIO}/molecular-profiles/{prof['mutation']}/mutations/fetch",
                         params={"projection": "SUMMARY"}, headers=JSON_HDR,
                         json={"entrezGeneIds": entrez_ids, "sampleIds": sample_ids}, timeout=300).json()
    for m in muts:
        if m["entrezGeneId"] in e2g and m["sampleId"] in sset:
            sets[e2g[m["entrezGeneId"]]].add(m["sampleId"])

    if alterations == "altered":
        # CNA: discrete GISTIC values, amplification (+2) or deep deletion (-2) only
        if "cna" in prof:
            cna = requests.post(f"{CBIO}/molecular-profiles/{prof['cna']}/molecular-data/fetch",
                                headers=JSON_HDR,
                                json={"entrezGeneIds": entrez_ids, "sampleIds": sample_ids}, timeout=300).json()
            for d in cna:
                if d.get("value") in (-2, 2) and d["entrezGeneId"] in e2g and d["sampleId"] in sset:
                    sets[e2g[d["entrezGeneId"]]].add(d["sampleId"])
        # structural variants / fusions: gene may be either breakpoint partner
        if "sv" in prof:
            sv = requests.post(f"{CBIO}/structural-variant/fetch", headers=JSON_HDR,
                               json={"molecularProfileIds": [prof["sv"]], "entrezGeneIds": entrez_ids},
                               timeout=300).json()
            for x in sv:
                sid = x.get("sampleId")
                if sid in sset:
                    for sym in (x.get("site1HugoSymbol"), x.get("site2HugoSymbol")):
                        if sym in sets:
                            sets[sym].add(sid)
    return sets

def get_binary_matrix(study_id, entrez_map, cohort, level="patient", alterations="mutation"):
    """Binary alteration matrix indexed by patient (default) or sample. A unit is '1'
    for a gene if any of its samples is altered (mutation, or mutation/CNA/SV if 'altered')."""
    sample_ids = cohort["sampleId"].tolist()
    sets = _altered_sample_sets(study_id, entrez_map, sample_ids, alterations)
    s2p = dict(zip(cohort["sampleId"], cohort["patientId"]))
    genes = list(entrez_map)

    if level == "patient":
        index_ids = cohort["patientId"].drop_duplicates().tolist()
        to_unit = lambda s: s2p.get(s)
    else:
        index_ids = sample_ids
        to_unit = lambda s: s

    bm = pd.DataFrame(0, index=index_ids, columns=genes)
    for g in genes:
        units = {to_unit(s) for s in sets[g]} - {None}
        if units:
            bm.loc[list(units), g] = 1
    return bm

In [19]:
all_genes = list(dict.fromkeys(GENES_X + GENES_Y))     # union, order-preserving
entrez_map = resolve_genes(all_genes)

cohort = get_cohort(STUDY_ID, CANCER_TYPE)
bm = get_binary_matrix(STUDY_ID, entrez_map, cohort, level=LEVEL, alterations=ALTERATIONS)
n_units = bm.shape[0]
unit = "patients" if LEVEL == "patient" else "samples"
print(f"Cohort: {STUDY_ID}" + (f"  /  {CANCER_TYPE}" if CANCER_TYPE else "  /  (whole study)")
      + f"  ->  {len(cohort)} samples, {cohort['patientId'].nunique()} patients"
      + f"   (analyzing {ALTERATIONS} at {LEVEL} level: n={n_units})")

# valid genes = resolved AND present after matrix build; warn on zero-alteration genes
gx = [g for g in GENES_X if g in bm.columns]
gy = [g for g in GENES_Y if g in bm.columns]
zero = [g for g in bm.columns if bm[g].sum() == 0]
if zero:
    print(f"NOTE: 0 alterations in cohort for {zero} (not on panel, or simply unaltered here).")
print(f"\nPer-gene {ALTERATIONS} frequency in this cohort ({LEVEL}-level):")
display((bm.sum().sort_values(ascending=False) / n_units * 100).round(1).to_frame("freq_%"))

Cohort: msk_impact_50k_2026  /  Lung Adenocarcinoma  ->  6254 samples, 5224 patients   (analyzing altered at sample level: n=6254)

Per-gene altered frequency in this cohort (sample-level):


,freq_%
TP53,50.1
EGFR,33.0
KRAS,32.4
STK11,16.6
KEAP1,14.4
BRAF,5.9


## 3. Co-occurrence statistics (Fisher's exact + BH-FDR)

For each gene pair we build the 2×2 table — **Both**, **A-not-B**, **B-not-A**, **Neither** — and run a Fisher's exact test.
The **log₂ odds ratio** uses the plain formula `log2((Both·Neither)/(A_not_B·B_not_A))` (matching cBioPortal); a Haldane +0.5
correction is applied only when a cell is empty, to keep it finite. q-values are BH-FDR-adjusted across all unique pairs.

With `LEVEL="sample"` and `ALTERATIONS="altered"`, this table reproduces cBioPortal's **Mutual Exclusivity** tab.

In [20]:
def cooccurrence(bm, genes_x, genes_y):
    count  = pd.DataFrame(0.0,    index=genes_y, columns=genes_x)
    log2or = pd.DataFrame(np.nan, index=genes_y, columns=genes_x)
    pval   = pd.DataFrame(np.nan, index=genes_y, columns=genes_x)
    pairs, table = [], {}
    for cx in genes_x:
        for cy in genes_y:
            x, y = bm[cx], bm[cy]
            a = int(((x == 1) & (y == 1)).sum())      # both
            count.loc[cy, cx] = a
            if cx == cy:
                continue
            b = int(((x == 1) & (y == 0)).sum())      # x only
            c = int(((x == 0) & (y == 1)).sum())      # y only
            d = int(((x == 0) & (y == 0)).sum())      # neither
            _, p = fisher_exact([[a, b], [c, d]])
            # plain odds ratio (matches cBioPortal); Haldane +0.5 only if a cell is empty (else 0/inf)
            if min(a, b, c, d) == 0:
                lor = np.log2(((a + 0.5) * (d + 0.5)) / ((b + 0.5) * (c + 0.5)))
            else:
                lor = np.log2((a * d) / (b * c))
            log2or.loc[cy, cx] = lor
            pval.loc[cy, cx] = p
            pairs.append((frozenset((cx, cy)), p))
            table[frozenset((cx, cy))] = (a, b, c, d)

    # BH-FDR over unique unordered pairs
    uniq = dict(pairs)
    qmap = {}
    if uniq:
        keys = list(uniq)
        qs = multipletests([uniq[k] for k in keys], method="fdr_bh")[1]
        qmap = dict(zip(keys, qs))
    qval = pd.DataFrame(np.nan, index=genes_y, columns=genes_x)
    for cx in genes_x:
        for cy in genes_y:
            if cx != cy:
                qval.loc[cy, cx] = qmap.get(frozenset((cx, cy)), np.nan)
    return count, log2or, pval, qval, table

count, log2or, pval, qval, table = cooccurrence(bm, gx, gy)

# tidy 2x2 + stats table (mirrors cBioPortal's Mutual Exclusivity layout)
rows = []
for cx in gx:
    for cy in gy:
        if cx < cy:  # each unordered pair once
            a, b, c, d = table[frozenset((cx, cy))]   # both, A-not-B, B-not-A, neither
            rows.append({"A": cx, "B": cy, "Neither": d, "A_not_B": b, "B_not_A": c, "Both": a,
                         "log2_OR": log2or.loc[cy, cx], "p": pval.loc[cy, cx], "q": qval.loc[cy, cx],
                         "tendency": "co-occur" if log2or.loc[cy, cx] > 0 else "exclusive"})
pairs_df = pd.DataFrame(rows).sort_values("q")
print(f"Pairwise results ({ALTERATIONS}, {LEVEL}-level, n={n_units}; sorted by q):")
display(pairs_df.round(3))

Pairwise results (altered, sample-level, n=6254; sorted by q):


,A,B,Neither,A_not_B,B_not_A,Both,log2_OR,p,q,tendency
3,EGFR,KRAS,2246,1979,1946,83,-4.369,0.000,0.000,exclusive
14,KEAP1,STK11,4863,352,489,550,3.958,0.000,0.000,co-occur
5,EGFR,STK11,3213,2002,979,60,-3.346,0.000,0.000,exclusive
6,EGFR,KEAP1,3350,842,2002,60,-3.068,0.000,0.000,exclusive
1,KRAS,STK11,3744,481,1471,558,1.562,0.000,0.000,co-occur
0,KRAS,TP53,1843,2382,1275,754,-1.128,0.000,0.000,exclusive
4,EGFR,TP53,2309,809,1883,1253,0.925,0.000,0.000,co-occur
2,STK11,TP53,2458,660,2757,379,-0.966,0.000,0.000,exclusive
12,KEAP1,KRAS,3738,487,1614,415,0.981,0.000,0.000,co-occur
10,BRAF,EGFR,3881,311,2005,57,-1.495,0.000,0.000,exclusive


## 4. Side-by-side heatmaps (counts + significance-colored odds ratio)

In [21]:
def cooccurrence_figure(count, log2or, qval, study_id, cancer_type, n, unit="patients",
                        alterations="mutation", sig=0.05):
    title_ct = cancer_type or "whole study"
    star = qval.map(lambda q: "★" if (pd.notna(q) and q < sig) else "")

    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.16,
                        subplot_titles=(f"Co-occurrence count (both {alterations})",
                                        f"log₂ odds ratio  (★ = FDR < {sig})"))
    # left: raw counts
    fig.add_trace(go.Heatmap(
        z=count.values, x=count.columns.tolist(), y=count.index.tolist(),
        colorscale="Blues", text=count.values.astype(int), texttemplate="%{text}",
        colorbar=dict(title=f"# {unit}", x=0.43, len=0.9),
        hovertemplate="%{x} & %{y}<br>both = %{z}<extra></extra>"), row=1, col=1)
    # right: log2 OR, diverging & zero-centered; red = co-occur, blue = exclusive
    fig.add_trace(go.Heatmap(
        z=log2or.values, x=log2or.columns.tolist(), y=log2or.index.tolist(),
        colorscale="RdBu_r", zmid=0, text=star.values, texttemplate="%{text}",
        textfont=dict(size=18),
        customdata=np.dstack([count.values, qval.values]),
        colorbar=dict(title="log₂ OR", x=1.0, len=0.9),
        hovertemplate="%{x} & %{y}<br>log2 OR = %{z:.2f}<br>both = %{customdata[0]:.0f}"
                      "<br>q = %{customdata[1]:.3g}<extra></extra>"), row=1, col=2)

    fig.update_layout(height=520, width=1180,
                      title_text=f"Mutational co-occurrence / exclusivity — {study_id} · {title_ct} "
                                 f"({alterations}, n={n} {unit})")
    fig.update_yaxes(autorange="reversed")
    return fig

cooccurrence_figure(count, log2or, qval, STUDY_ID, CANCER_TYPE, n_units,
                    unit=unit, alterations=ALTERATIONS).show()

## Notes & caveats

* **Two toggles.**
  * `ALTERATIONS = "altered"` counts a gene as hit by **mutation OR CNA (amp/deep-del, ±2) OR structural variant/fusion** — what cBioPortal calls "altered". `"mutation"` restricts to somatic mutations only.
  * `LEVEL = "sample"` counts each tumor sample; `"patient"` collapses a patient's samples (altered if *any* sample is).
  * **`LEVEL="sample"` + `ALTERATIONS="altered"` reproduces cBioPortal's Mutual Exclusivity tab exactly** (same 2×2, same log₂ OR). For a cleaner standalone analysis (independent cases, drivers only) use `LEVEL="patient"` + `ALTERATIONS="mutation"`.
* **Odds ratio.** Plain `log2((Both·Neither)/(A_not_B·B_not_A))` to match cBioPortal; Haldane +0.5 only when a cell is empty (keeps near-exclusive pairs finite).
* **Cohort is explicit.** `CANCER_TYPE = "Lung Adenocarcinoma"` filters to LUAD; `None` = whole study. `list_cancer_types(STUDY_ID)` shows valid values. (Selecting lung *genes* does not restrict samples — the original bug.)
* **Counts vs statistic.** Left panel = raw overlap (inflated for individually-frequent genes); right panel = odds ratio + Fisher test, which distinguishes real co-occurrence/exclusivity from a frequency artifact.
* **Two axes / any genes.** `GENES_X` and `GENES_Y` may be any length and need not match; unknown symbols are dropped with a warning, and genes with 0 alterations are flagged.
* **Portability.** CNA/SV profiles are auto-discovered per study; if a study lacks them, `"altered"` quietly falls back to whatever profiles exist (mutations always).